In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
import logging
import os
import re
import sys
import numpy as np
from itertools import chain
from gensim.models import KeyedVectors
import gensim
import pandas as pd
import torch
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
import pickle

# =================== 超参（和你本地保持一致） ===================
embed_size = 300
max_len = 512

# =================== Kaggle路径【自行核对修改】 ===================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
GLOVE_PATH = "/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt"

# =================== 文本清洗函数（原版不动） ===================
def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    return words

def encode_samples(tokenized_samples, word_to_idx):
    features = []
    for sample in tokenized_samples:
        feature = []
        for token in sample:
            if token in word_to_idx:
                feature.append(word_to_idx[token])
            else:
                feature.append(0)
        features.append(feature)
    return features

def pad_samples(features, maxlen=max_len, PAD=0):
    padded_features = []
    for feature in features:
        if len(feature) >= maxlen:
            padded_feature = feature[:maxlen]
        else:
            padded_feature = feature.copy()
            while len(padded_feature) < maxlen:
                padded_feature.append(PAD)
        padded_features.append(padded_feature)
    return padded_features

# =================== 主流程 ===================
os.makedirs("/kaggle/working/pickle", exist_ok=True)

train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

clean_train_reviews, train_labels = [], []
for i, review in enumerate(train["review"]):
    clean_train_reviews.append(review_to_wordlist(review))
    train_labels.append(train["sentiment"][i])

clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review))

vocab = set(chain(*clean_train_reviews)) | set(chain(*clean_test_reviews))
vocab_size = len(vocab)

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    clean_train_reviews, train_labels, test_size=0.2, random_state=0)

# ===================【重点修改】适配Common Crawl 840B（glove-gensim分割逻辑） ===================
wvmodel = KeyedVectors(embed_size)
word_dict = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tokens = line.split()
        if len(tokens) <= embed_size:
            continue
        try:
            vec = np.array(tokens[-embed_size:], dtype=np.float32)
            word = " ".join(tokens[:-embed_size])
            word_dict[word] = vec
        except ValueError:
            continue
wvmodel.add_vectors(list(word_dict.keys()), list(word_dict.values()))
print(f"GloVe加载完成，载入词总数：{len(wvmodel)}")
# =========================================================================================

word_to_idx = {word: i + 1 for i, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0
idx_to_word = {i + 1: word for i, word in enumerate(vocab)}
idx_to_word[0] = '<unk>'

train_features = torch.tensor(pad_samples(encode_samples(train_reviews, word_to_idx)))
val_features = torch.tensor(pad_samples(encode_samples(val_reviews, word_to_idx)))
test_features = torch.tensor(pad_samples(encode_samples(clean_test_reviews, word_to_idx)))

train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 构建Embedding权重矩阵
weight = torch.zeros(vocab_size + 1, embed_size)
hit = 0
for word, idx in word_to_idx.items():
    if word in wvmodel:
        weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))
        hit += 1
print(f"词表匹配成功向量：{hit}/{len(word_to_idx)}")

pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
pickle.dump(
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab],
    open(pickle_file, 'wb'))
print('pickle文件生成完成！')

GloVe加载完成，载入词总数：2195895


/tmp/ipykernel_23/1117333043.py:114: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))


词表匹配成功向量：77554/101400
pickle文件生成完成！


In [3]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ===================== Kaggle环境配置 =====================
num_epochs = 10
embed_size = 300
num_filter = 128
filter_size = 3
bidirectional = True
batch_size = 64
labels = 2
lr = 0.8
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
use_gpu = torch.cuda.is_available()

# 自动创建输出文件夹
os.makedirs("/kaggle/working/result", exist_ok=True)

# ===================== CNN网络定义 =====================
class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_size, weight, labels, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.use_gpu = use_gpu
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False

        self.conv1d = nn.Conv1d(embed_size, num_filter, filter_size, padding=1)
        self.decoder = nn.Linear(num_filter, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        # [batch, seq_len, embed] -> [batch, embed, seq_len] 适配Conv1d
        convolution = F.relu(self.conv1d(embeddings.permute([0, 2, 1])))
        pooling = F.max_pool1d(convolution, kernel_size=convolution.shape[2])
        outputs = self.decoder(pooling.squeeze(dim=2))
        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {''.join(sys.argv)}")

    logging.info('loading data...')
    # 路径和你前面imdb_process输出严格对齐
    pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size, num_filter=num_filter, filter_size=filter_size,
                       weight=weight, labels=labels, use_gpu=use_gpu)
    net.to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0.0, 0.0
        train_acc, val_acc = 0.0, 0.0
        n, m = 0, 0
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch}') as pbar:
            for feature, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                optimizer.step()

                train_acc += accuracy_score(torch.argmax(score.cpu().data, dim=1), label.cpu())
                train_loss += loss.item()

                pbar.set_postfix({
                    'train loss': f'{train_loss / n:.4f}',
                    'train acc': f'{train_acc / n:.2f}'
                })
                pbar.update(1)

            with torch.no_grad():
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label)
                    val_acc += accuracy_score(torch.argmax(val_score.cpu().data, dim=1), val_label.cpu())
                    val_losses += val_loss.item()

        end = time.time()
        runtime = end - start
        pbar.set_postfix({
            'train loss': f'{train_loss / n:.4f}',
            'train acc': f'{train_acc / n:.2f}',
            'val loss': f'{val_losses / m:.4f}',
            'val acc': f'{val_acc / m:.2f}',
            'time': f'{runtime:.2f}'
        })

    # 预测测试集
    test_pred = []
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    # 读取原始测试文件生成提交csv（Kaggle路径）
    TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)
    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv("/kaggle/working/result/cnn.csv", index=False, quoting=3)
    logging.info('result saved!')

INFO:colab_kernel_launcher.py:running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/tmp/tmpvmnhde_n.json--HistoryManager.hist_file=:memory:
INFO:root:loading data...
INFO:root:data loaded!
Prediction: 100%|██████████| 391/391 [00:01<00:00, 297.59it/s]
INFO:root:result saved!
